In [1]:
import sys
import subprocess
import site
import importlib

print("Hugging Face ve Çeviri kütüphaneleri kuruluyor...")
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'datasets', 'deep-translator', 'pandas', 'tqdm'])
    
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
        
    importlib.invalidate_caches()
    print("✅ Hazırlık tamamlandı!")
except Exception as e:
    print(f"❌ Kurulum hatası: {e}")

Hugging Face ve Çeviri kütüphaneleri kuruluyor...
✅ Hazırlık tamamlandı!


In [3]:
import os
import time
import pandas as pd
from datasets import load_dataset
from deep_translator import GoogleTranslator
from tqdm.notebook import tqdm

# Ayarlar
DATASET_NAME = "abdelhakimDZ/diabetes_QA_dataset"
OUTPUT_FILE = "data/07_hf_diabetes_tr.csv"

# 1. Veriyi Hugging Face'den Çek
print(f"'{DATASET_NAME}' veri seti indiriliyor...")
dataset = load_dataset(DATASET_NAME)
df_raw = dataset['train'].to_pandas()

print(f"Veri setindeki orijinal sütunlar: {df_raw.columns.tolist()}")

# 2. Akıllı Sütun Yakalayıcı (Büyük/Küçük Harf Duyarsız)
q_col = next((col for col in df_raw.columns if 'question' in col.lower()), df_raw.columns[0])
a_col = next((col for col in df_raw.columns if 'answer' in col.lower()), df_raw.columns[1])

print(f"✅ Kilitlenen Soru Sütunu: '{q_col}' | Cevap Sütunu: '{a_col}'\n")

# 3. Kaldığı Yerden Devam Etme (Checkpoint) Kontrolü
if os.path.exists(OUTPUT_FILE):
    df_existing = pd.read_csv(OUTPUT_FILE)
    processed_questions = set(df_existing['question_en'].tolist())
    print(f"Daha önce {len(df_existing)} satır çevrilmiş. Kaldığı yerden devam ediliyor...")
else:
    # Dosya yoksa sütunlarla oluştur
    pd.DataFrame(columns=["source", "question_tr", "answer_tr", "question_en"]).to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    processed_questions = set()

# 4. Çeviri Motoru
translator = GoogleTranslator(source='en', target='tr')
df_to_process = df_raw[~df_raw[q_col].isin(processed_questions)]

print(f"Çevrilecek {len(df_to_process)} yeni satır bulundu. Çeviri başlıyor...\n")

new_rows = []
try:
    with tqdm(total=len(df_to_process), desc="Diyabet Çevirisi") as pbar:
        for _, row in df_to_process.iterrows():
            try:
                # Dinamik olarak bulunan sütunlardan veriyi çek
                q_en = str(row[q_col])
                a_en = str(row[a_col])
                
                # Türkçeye çevir
                q_tr = translator.translate(q_en)
                a_tr = translator.translate(a_en)
                
                new_rows.append({
                    "source": "HF_Diabetes_TR_Translated",
                    "question_tr": q_tr,
                    "answer_tr": a_tr,
                    "question_en": q_en
                })
                
                # Her 15 satırda bir diske yaz (Güvenlik önlemi)
                if len(new_rows) >= 15:
                    pd.DataFrame(new_rows).to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8-sig")
                    new_rows = []
                
                time.sleep(0.4) # Google ban riskini azaltmak için küçük bekleme
                pbar.update(1)
                
            except Exception as e:
                # 429 hatası Google'ın bizi yavaşlatmasıdır, panik yok
                if "429" in str(e):
                    print("\n[Hız Sınırı] Google geçici olarak durdurdu. 60 saniye dinleniyoruz...")
                    time.sleep(60)
                continue

except KeyboardInterrupt:
    print("\nİşlem kullanıcı tarafından durduruldu. O ana kadar olan veriler kaydedildi.")

# Kalan son verileri yaz
if new_rows:
    pd.DataFrame(new_rows).to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8-sig")

print(f"\n🎉 İşlem bitti! Yeni veri setin hazır: {os.path.abspath(OUTPUT_FILE)}")

'abdelhakimDZ/diabetes_QA_dataset' veri seti indiriliyor...
Veri setindeki orijinal sütunlar: ['question', 'answer']
✅ Kilitlenen Soru Sütunu: 'question' | Cevap Sütunu: 'answer'

Daha önce 0 satır çevrilmiş. Kaldığı yerden devam ediliyor...
Çevrilecek 1075 yeni satır bulundu. Çeviri başlıyor...



Diyabet Çevirisi:   0%|          | 0/1075 [00:00<?, ?it/s]


🎉 İşlem bitti! Yeni veri setin hazır: e:\LLM-FULLPARAMETER\data\07_hf_diabetes_tr.csv
